# Resolució de problemes de cerca

En aquest quadern representarem els estats amb estructures senzilles i buscarem una ruta fins a l'objectiu. Ho farem amb dos exemples: les torres de Hanoi i un trajecte per Castelló.

## Torres de Hanoi

### Representació de l'estat

Un estat és una tupla de tres torres. Cada torre és una tupla de discos ordenats de baix cap amunt. Per exemple, `((3, 2, 1), (), ())` indica que els tres discos són a la primera torre. Les tuples es poden guardar en un conjunt de visitats i en un diccionari de pares.

In [ ]:
def genera_estats(estat):
    estats = []

    # Prova a moure el disc superior de cada torre a una altra torre.
    for origen in range(3):
        if not estat[origen]:
            continue

        disc = estat[origen][-1]
        for desti in range(3):
            if origen == desti:
                continue

            # Només es pot posar un disc sobre una torre buida
            # o sobre un disc més gran.
            if not estat[desti] or disc < estat[desti][-1]:
                nou_estat = [list(torre) for torre in estat]
                nou_estat[desti].append(nou_estat[origen].pop())
                estats.append(tuple(tuple(torre) for torre in nou_estat))

    return estats

La funció següent busca una solució amb **BFS** (cerca en amplària). Com que cada moviment costa un pas, BFS troba una solució amb el nombre mínim de moviments. El diccionari `pares` recorda des de quin estat hem arribat a cadascun.

In [ ]:
from collections import deque

def bfs(inicial, es_objectiu, genera_estats):
    cua = deque([inicial])
    visitats = set()
    pares = {inicial: None}

    while cua:
        estat = cua.popleft()
        if estat in visitats:
            continue
        visitats.add(estat)
        if es_objectiu(estat):
            return estat, pares

        for successor in genera_estats(estat):
            if successor not in visitats:
                if successor not in pares:
                    pares[successor] = estat
                cua.append(successor)

    return None, pares


def reconstrueix_ruta(pares, final):
    ruta = []
    while final is not None:
        ruta.append(final)
        final = pares[final]
    return list(reversed(ruta))

### Resolució

Comencem amb tres discos. La funció objectiu comprova si tots han arribat a la tercera torre.

In [ ]:
inicial = ((3, 2, 1), (), ())
final, pares = bfs(
    inicial,
    lambda estat: estat == ((), (), (3, 2, 1)),
    genera_estats
)

for pas, estat in enumerate(reconstrueix_ruta(pares, final)):
    print(f"Pas {pas}: {estat}")

## Ruta per Castelló

Ara cada estat és el nom d'un lloc. El graf guarda els llocs als quals podem anar i el cost de cada trajecte. A diferència de Hanoi, ací els trajectes poden tindre costos diferents.

Les connexions, els costos i les estimacions són inventats per a l’exemple; no representen distàncies reals.

In [ ]:
# Cada aresta indica el lloc veí i el cost del trajecte.
graf = {
    "UJI": [("Borrull", 4), ("Agustina", 6)],
    "Borrull": [("UJI", 4), ("Agustina", 2)],
    "Agustina": [("UJI", 6), ("Borrull", 2), ("Santa Clara", 3)],
    "Santa Clara": [("Agustina", 3), ("Major", 2), ("Ribalta", 8)],
    "Major": [("Santa Clara", 2), ("Ribalta", 5)],
    "Ribalta": [("Santa Clara", 8), ("Major", 5)],
}

### Cerca de cost uniforme i A*

La funció manté el cost més barat conegut per a cada lloc i els seus pares per reconstruir la ruta.

- Amb una heurística de zeros, prioritza el cost recorregut: és la **cerca de cost uniforme**.
- Amb una estimació del cost que falta, prioritza `cost recorregut + estimació`: és **A\***.

Perquè A* trobe una ruta òptima, l'estimació no ha de superar mai el cost real que falta.

In [ ]:
import heapq


def cerca_cost_uniforme_o_a_estrella(inicial, final, heuristica):
    cua = [(heuristica[inicial], 0, inicial)]
    costos = {inicial: 0}
    pares = {inicial: None}

    while cua:
        prioritat, cost, estat = heapq.heappop(cua)

        # Ignora una entrada antiga si ja hem trobat un camí més barat.
        if cost != costos[estat]:
            continue

        if estat == final:
            return pares, cost

        for successor, cost_accio in graf[estat]:
            nou_cost = cost + cost_accio
            if successor not in costos or nou_cost < costos[successor]:
                costos[successor] = nou_cost
                pares[successor] = estat
                prioritat = nou_cost + heuristica[successor]
                heapq.heappush(cua, (prioritat, nou_cost, successor))

    return None, None


def mostra_ruta(pares, final):
    ruta = reconstrueix_ruta(pares, final)
    print(" → ".join(ruta))

### Cerca de cost uniforme

Amb una heurística de zeros, trobem la ruta de cost mínim des de l'UJI fins al parc Ribalta.

In [ ]:
inicial = "UJI"
final = "Ribalta"
heuristica_zeros = {lloc: 0 for lloc in graf}

pares, cost = cerca_cost_uniforme_o_a_estrella(inicial, final, heuristica_zeros)
mostra_ruta(pares, final)
print("Cost total:", cost)

### A* amb una estimació

Definim una estimació del cost que falta per arribar a la Ribalta. En aquest mapa és admissible: no supera el cost real més barat des de cada lloc.

In [ ]:
heuristica = {
    "UJI": 10,
    "Borrull": 8,
    "Agustina": 6,
    "Santa Clara": 4,
    "Major": 3,
    "Ribalta": 0,
}

pares, cost = cerca_cost_uniforme_o_a_estrella(inicial, final, heuristica)
mostra_ruta(pares, final)
print("Cost total:", cost)

En tots dos casos obtenim una ruta de cost mínim. La diferència és l'ordre d'exploració: A* utilitza l'estimació per orientar la cerca cap a l'objectiu.